# NYC 311 Schema and Data-Quality Analysis

## Purpose

This notebook validates and classifies the fields needed to assess the
Month 1 use case:

> Predict whether a newly created NYC 311 complaint will miss its
> expected resolution target.

The proposed target is based on comparing `closed_date` with `due_date`.
This notebook evaluates whether that analysis is feasible, but it does
**not** create the target, select final features, clean or remove records,
impute values, or train a model.

## 1. Imports and project paths

Only the existing notebook dependencies and standard-library HTTP tools
are used. Output paths come from the repository's shared path module, and
report directories are created idempotently.

In [1]:
from datetime import datetime, timezone
from http.client import RemoteDisconnected
from pathlib import Path
from urllib.error import HTTPError, URLError
from urllib.parse import urlencode
from urllib.request import Request, urlopen
import json
import socket
import sys
import time

import pandas as pd
from IPython.display import Markdown, display

pd.set_option("display.max_columns", 20)
pd.set_option("display.max_rows", 100)
pd.set_option("display.max_colwidth", 140)

PROJECT_ROOT = next(
    candidate
    for candidate in [Path.cwd(), *Path.cwd().parents]
    if (candidate / "src" / "urban_ops").is_dir()
)
SRC_DIR = PROJECT_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from urban_ops.data.nyc_311_config import (
    API_ENDPOINT,
    API_MAX_ATTEMPTS,
    API_TIMEOUT_SECONDS,
)
from urban_ops.utils.paths import (
    ensure_report_directories,
    notebook_report_paths,
)

REPORT_DIR, REPORT_TABLES_DIR, REPORT_FIGURES_DIR = notebook_report_paths(
    "02_schema_and_quality_analysis"
)
ensure_report_directories("02_schema_and_quality_analysis")
DATASET_ID = API_ENDPOINT.rsplit("/", maxsplit=1)[-1].removesuffix(".json")
DATASET_METADATA_URL = (
    f"https://data.cityofnewyork.us/api/views/{DATASET_ID}"
)
QUALITY_REPORT_PATH = REPORT_DIR / "data_quality_report.md"

PROJECT_ROOT, DATASET_ID, QUALITY_REPORT_PATH.relative_to(PROJECT_ROOT)

(PosixPath('/Users/mohammadmubashir/VCode/urban-operations-intelligence-platform'),
 'erm2-nwe9',
 PosixPath('reports/02_schema_and_quality_analysis/data_quality_report.md'))

## 2. Column roles and requirement levels

The selected columns are not based on a universal schema rule. They are
selected because of the current business question and the requirements
of the analysis pipeline. Not all listed columns are mandatory in the
same way:

- **Core required columns** provide the minimum complaint-level structure.
- **Target-construction columns** are needed to evaluate whether the
  proposed missed-resolution target can be defined.
- **Candidate feature columns** are possible creation-time predictors or
  subgroup fields, not guaranteed model inputs.
- **Supporting columns** improve reporting and consistency checks.

Target-construction fields are not automatically model features.
`closed_date` and final `status` contain post-creation information and
would leak outcome information if used for prediction. Whether `due_date`
exists at complaint creation must be confirmed before it can be considered
prediction-time information.

Candidate features will be assessed later for prediction-time
availability, leakage, missingness, stability, and predictive value. This
notebook does not decide the final feature list.

In [2]:
CORE_REQUIRED_COLUMNS = [
    "unique_key",
    "created_date",
    "agency",
    "complaint_type",
]

TARGET_CONSTRUCTION_COLUMNS = [
    "closed_date",
    "due_date",
    "status",
]

CANDIDATE_FEATURE_COLUMNS = [
    "agency",
    "complaint_type",
    "descriptor",
    "borough",
    "open_data_channel_type",
]

SUPPORTING_COLUMNS = [
    "agency_name",
]

COLUMN_ROLE_RECORDS = [
    {
        "column_name": "unique_key",
        "column_group": "core dataset requirement",
        "role": "identifier",
        "why_needed": (
            "Duplicate detection and complaint-level traceability."
        ),
        "required_for_current_stage": "yes",
        "available_at_prediction_time": "yes",
        "potential_leakage": "no",
    },
    {
        "column_name": "created_date",
        "column_group": "core dataset requirement",
        "role": "event timestamp",
        "why_needed": (
            "Prediction moment, timestamp validation, and chronological splitting."
        ),
        "required_for_current_stage": "yes",
        "available_at_prediction_time": "yes",
        "potential_leakage": "no",
    },
    {
        "column_name": "closed_date",
        "column_group": "target construction",
        "role": "outcome timestamp",
        "why_needed": "Compare actual closure with the due date.",
        "required_for_current_stage": (
            "needed for target feasibility analysis"
        ),
        "available_at_prediction_time": "no",
        "potential_leakage": "yes, if used as a model feature",
    },
    {
        "column_name": "due_date",
        "column_group": "target construction",
        "role": "expected resolution deadline",
        "why_needed": "Define whether the complaint missed its target.",
        "required_for_current_stage": (
            "needed for target feasibility analysis"
        ),
        "available_at_prediction_time": "must be confirmed",
        "potential_leakage": (
            "depends on whether it exists at complaint creation"
        ),
    },
    {
        "column_name": "status",
        "column_group": "target construction",
        "role": "target-supporting workflow state",
        "why_needed": (
            "Detect contradictions such as Closed status without a closed date."
        ),
        "required_for_current_stage": (
            "needed for target feasibility quality checks"
        ),
        "available_at_prediction_time": (
            "initial status may be available; final status is not"
        ),
        "potential_leakage": "yes, when final status is used",
    },
    {
        "column_name": "agency",
        "column_group": "core requirement and candidate feature",
        "role": "responsible organization",
        "why_needed": (
            "Scope selection, subgroup quality analysis, and potential modelling."
        ),
        "required_for_current_stage": "yes",
        "available_at_prediction_time": "yes",
        "potential_leakage": "no",
    },
    {
        "column_name": "agency_name",
        "column_group": "supporting metadata",
        "role": "human-readable agency label",
        "why_needed": "Agency-code mapping validation and reporting.",
        "required_for_current_stage": "no",
        "available_at_prediction_time": "usually yes",
        "potential_leakage": "no",
    },
    {
        "column_name": "complaint_type",
        "column_group": "core requirement and candidate feature",
        "role": "main request category",
        "why_needed": (
            "Scope analysis, category baselines, and potential modelling."
        ),
        "required_for_current_stage": "yes",
        "available_at_prediction_time": "yes",
        "potential_leakage": "no",
    },
    {
        "column_name": "descriptor",
        "column_group": "candidate feature",
        "role": "detailed complaint category",
        "why_needed": (
            "Additional operational detail and possible NLP or categorical modelling."
        ),
        "required_for_current_stage": "no",
        "available_at_prediction_time": "generally yes",
        "potential_leakage": "no, subject to source validation",
    },
    {
        "column_name": "borough",
        "column_group": "candidate feature",
        "role": "geographic grouping",
        "why_needed": (
            "Subgroup analysis and location-related modelling."
        ),
        "required_for_current_stage": "no",
        "available_at_prediction_time": "generally yes",
        "potential_leakage": "no",
    },
    {
        "column_name": "open_data_channel_type",
        "column_group": "candidate feature",
        "role": "complaint intake channel",
        "why_needed": (
            "Quality analysis and possible prediction-time signal."
        ),
        "required_for_current_stage": "no",
        "available_at_prediction_time": "yes",
        "potential_leakage": "no",
    },
]

column_role_summary = pd.DataFrame(COLUMN_ROLE_RECORDS)
column_role_summary.to_csv(
    REPORT_TABLES_DIR / "column_role_summary.csv",
    index=False,
)
column_role_summary

,column_name,column_group,role,why_needed,required_for_current_stage,available_at_prediction_time,potential_leakage
0,unique_key,core dataset requirement,identifier,Duplicate detection and complaint-level traceability.,yes,yes,no
1,created_date,core dataset requirement,event timestamp,"Prediction moment, timestamp validation, and chronological splitting.",yes,yes,no
2,closed_date,target construction,outcome timestamp,Compare actual closure with the due date.,needed for target feasibility analysis,no,"yes, if used as a model feature"
3,due_date,target construction,expected resolution deadline,Define whether the complaint missed its target.,needed for target feasibility analysis,must be confirmed,depends on whether it exists at complaint creation
4,status,target construction,target-supporting workflow state,Detect contradictions such as Closed status without a closed date.,needed for target feasibility quality checks,initial status may be available; final status is not,"yes, when final status is used"
5,agency,core requirement and candidate feature,responsible organization,"Scope selection, subgroup quality analysis, and potential modelling.",yes,yes,no
6,agency_name,supporting metadata,human-readable agency label,Agency-code mapping validation and reporting.,no,usually yes,no
7,complaint_type,core requirement and candidate feature,main request category,"Scope analysis, category baselines, and potential modelling.",yes,yes,no
8,descriptor,candidate feature,detailed complaint category,Additional operational detail and possible NLP or categorical modelling.,no,generally yes,"no, subject to source validation"
9,borough,candidate feature,geographic grouping,Subgroup analysis and location-related modelling.,no,generally yes,no


## 3. Read-only schema and completeness inspection

Dataset metadata identifies source fields without manufacturing absent
columns through dataframe reindexing. Server-side counts assess whether
important fields contain any values. Timestamp usability is an
analysis-time signal based on a bounded, deterministic non-null sample;
no values are changed.

In [3]:
def fetch_json_url(url: str) -> object:
    """Return JSON from a read-only request with bounded retries."""
    request = Request(
        url,
        headers={
            "Accept": "application/json",
            "User-Agent": (
                "urban-operations-intelligence-schema-quality/1.0"
            ),
        },
    )
    last_error: Exception | None = None

    for attempt in range(1, API_MAX_ATTEMPTS + 1):
        try:
            with urlopen(request, timeout=API_TIMEOUT_SECONDS) as response:
                return json.load(response)
        except (
            HTTPError,
            URLError,
            TimeoutError,
            socket.timeout,
            RemoteDisconnected,
            json.JSONDecodeError,
        ) as error:
            last_error = error
            if attempt < API_MAX_ATTEMPTS:
                time.sleep(2 ** (attempt - 1))

    raise RuntimeError(
        f"Read-only request failed after {API_MAX_ATTEMPTS} attempts."
    ) from last_error


def fetch_api_records(soql_query: str) -> list[dict[str, object]]:
    """Return a list of records for a read-only SoQL query."""
    query_url = f"{API_ENDPOINT}?{urlencode({'$query': soql_query})}"
    payload = fetch_json_url(query_url)
    if not isinstance(payload, list):
        raise RuntimeError("NYC Open Data returned JSON that was not a list.")
    return payload


metadata_payload = fetch_json_url(DATASET_METADATA_URL)
if not isinstance(metadata_payload, dict):
    raise RuntimeError("Dataset metadata response was not a JSON object.")

metadata_columns = metadata_payload.get("columns")
if not isinstance(metadata_columns, list):
    raise RuntimeError("Dataset metadata did not include a column list.")

dataset_columns = {
    str(column["fieldName"])
    for column in metadata_columns
    if isinstance(column, dict) and column.get("fieldName")
}

IMPORTANT_COLUMNS = list(
    dict.fromkeys(
        CORE_REQUIRED_COLUMNS
        + TARGET_CONSTRUCTION_COLUMNS
        + CANDIDATE_FEATURE_COLUMNS
        + SUPPORTING_COLUMNS
    )
)
present_important_columns = [
    column for column in IMPORTANT_COLUMNS if column in dataset_columns
]

count_expressions = ["count(*) AS total_rows"] + [
    f"count({column}) AS {column}_non_null"
    for column in present_important_columns
]
count_record = fetch_api_records(
    "SELECT " + ", ".join(count_expressions)
)[0]
total_row_count = int(count_record["total_rows"])
non_null_counts = {
    column: int(count_record.get(f"{column}_non_null", 0))
    for column in IMPORTANT_COLUMNS
}

target_timestamp_columns = [
    column
    for column in ["closed_date", "due_date"]
    if column in dataset_columns and non_null_counts[column] > 0
]
target_parse_summary = pd.DataFrame(
    columns=[
        "column_name",
        "sample_non_null_count",
        "sample_parseable_count",
        "sample_unparseable_count",
        "has_usable_values",
    ]
)
if target_timestamp_columns:
    parse_rows = []
    for column in target_timestamp_columns:
        target_sample_query = (
            f"SELECT {column} WHERE {column} IS NOT NULL "
            "ORDER BY created_date, unique_key LIMIT 5000"
        )
        target_sample = pd.DataFrame(
            fetch_api_records(target_sample_query)
        ).reindex(columns=[column])
        source_values = target_sample[column].dropna()
        parsed_values = pd.to_datetime(source_values, errors="coerce")
        parse_rows.append(
            {
                "column_name": column,
                "sample_non_null_count": int(len(source_values)),
                "sample_parseable_count": int(parsed_values.notna().sum()),
                "sample_unparseable_count": int(parsed_values.isna().sum()),
                "has_usable_values": bool(parsed_values.notna().any()),
            }
        )
    target_parse_summary = pd.DataFrame(parse_rows)

retrieved_at_utc = datetime.now(timezone.utc)
target_parse_summary

,column_name,sample_non_null_count,sample_parseable_count,sample_unparseable_count,has_usable_values
0,closed_date,5000,5000,0,True
1,due_date,5000,5000,0,True


## 4. Requirement-aware column validation

Missing-column severity follows the role of the field. A missing
candidate feature limits only that candidate; it does not invalidate the
core complaint schema. The helper below is the single source of validation
logic for all four groups.

In [4]:
GROUP_VALIDATION_RULES = {
    "core_required": {
        "columns": CORE_REQUIRED_COLUMNS,
        "severity_if_missing": "critical",
        "impact_if_missing": (
            "The dataset cannot reliably support complaint-level analysis."
        ),
    },
    "target_construction": {
        "columns": TARGET_CONSTRUCTION_COLUMNS,
        "severity_if_missing": "critical_for_target",
        "impact_if_missing": (
            "General analysis may remain possible, but the current "
            "missed-resolution-target use case cannot be supported."
        ),
    },
    "candidate_feature": {
        "columns": CANDIDATE_FEATURE_COLUMNS,
        "severity_if_missing": "warning",
        "impact_if_missing": (
            "This specific modelling or subgroup-analysis option is reduced."
        ),
    },
    "supporting_metadata": {
        "columns": SUPPORTING_COLUMNS,
        "severity_if_missing": "informational",
        "impact_if_missing": (
            "Reporting or consistency-check capability is reduced."
        ),
    },
}


def validate_column_groups(
    available_columns: set[str],
    group_rules: dict[str, dict[str, object]],
) -> pd.DataFrame:
    """Return one validation row per classified column-role membership."""
    validation_rows = []
    for column_group, rule in group_rules.items():
        for column_name in rule["columns"]:
            validation_rows.append(
                {
                    "column_name": column_name,
                    "column_group": column_group,
                    "is_present": column_name in available_columns,
                    "severity_if_missing": rule["severity_if_missing"],
                    "impact_if_missing": rule["impact_if_missing"],
                }
            )
    return pd.DataFrame(validation_rows)


column_validation = validate_column_groups(
    dataset_columns,
    GROUP_VALIDATION_RULES,
)
column_validation.to_csv(
    REPORT_TABLES_DIR / "column_requirement_validation.csv",
    index=False,
)
column_validation

,column_name,column_group,is_present,severity_if_missing,impact_if_missing
0,unique_key,core_required,True,critical,The dataset cannot reliably support complaint-level analysis.
1,created_date,core_required,True,critical,The dataset cannot reliably support complaint-level analysis.
2,agency,core_required,True,critical,The dataset cannot reliably support complaint-level analysis.
3,complaint_type,core_required,True,critical,The dataset cannot reliably support complaint-level analysis.
4,closed_date,target_construction,True,critical_for_target,"General analysis may remain possible, but the current missed-resolution-target use case cannot be supported."
5,due_date,target_construction,True,critical_for_target,"General analysis may remain possible, but the current missed-resolution-target use case cannot be supported."
6,status,target_construction,True,critical_for_target,"General analysis may remain possible, but the current missed-resolution-target use case cannot be supported."
7,agency,candidate_feature,True,warning,This specific modelling or subgroup-analysis option is reduced.
8,complaint_type,candidate_feature,True,warning,This specific modelling or subgroup-analysis option is reduced.
9,descriptor,candidate_feature,True,warning,This specific modelling or subgroup-analysis option is reduced.


## 5. Separate schema and availability results

Each result answers a different question. There is intentionally no
single pass/fail flag that treats structural requirements, target inputs,
candidate features, and descriptive metadata as equivalent.

In [5]:
def missing_columns(columns: list[str]) -> list[str]:
    """Return configured columns absent from source metadata."""
    return [column for column in columns if column not in dataset_columns]


def target_column_has_usable_values(column: str) -> bool:
    """Return whether a target input is present and has usable values."""
    if column not in dataset_columns or non_null_counts[column] == 0:
        return False
    if column not in {"closed_date", "due_date"}:
        return True
    matches = target_parse_summary.loc[
        target_parse_summary["column_name"] == column,
        "has_usable_values",
    ]
    return bool(matches.iloc[0]) if not matches.empty else False


core_missing = missing_columns(CORE_REQUIRED_COLUMNS)
target_missing = missing_columns(TARGET_CONSTRUCTION_COLUMNS)
target_unusable = [
    column
    for column in TARGET_CONSTRUCTION_COLUMNS
    if column in dataset_columns
    and not target_column_has_usable_values(column)
]
candidate_missing = missing_columns(CANDIDATE_FEATURE_COLUMNS)
supporting_missing = missing_columns(SUPPORTING_COLUMNS)

core_status = "Passed" if not core_missing else "Failed"
target_status = (
    "Passed"
    if not target_missing and not target_unusable
    else "Failed"
)
candidate_status = (
    "Passed"
    if not candidate_missing
    else (
        "Failed"
        if len(candidate_missing) == len(CANDIDATE_FEATURE_COLUMNS)
        else "Partially passed"
    )
)
supporting_status = (
    "Passed" if not supporting_missing else "Partially passed"
)

schema_validation_summary = pd.DataFrame(
    [
        {
            "validation_area": "Core schema",
            "question": (
                "Are the minimum complaint-level columns available?"
            ),
            "status": core_status,
            "missing_columns": ", ".join(core_missing) or "None",
            "unusable_columns": "None",
        },
        {
            "validation_area": "Target feasibility inputs",
            "question": (
                "Are the fields needed to evaluate target feasibility available?"
            ),
            "status": target_status,
            "missing_columns": ", ".join(target_missing) or "None",
            "unusable_columns": ", ".join(target_unusable) or "None",
        },
        {
            "validation_area": "Candidate feature availability",
            "question": (
                "Which potential creation-time features are available?"
            ),
            "status": candidate_status,
            "missing_columns": ", ".join(candidate_missing) or "None",
            "unusable_columns": "Not assessed at this stage",
        },
        {
            "validation_area": "Supporting metadata",
            "question": (
                "Which descriptive or consistency-check columns are available?"
            ),
            "status": supporting_status,
            "missing_columns": ", ".join(supporting_missing) or "None",
            "unusable_columns": "Not assessed at this stage",
        },
    ]
)
schema_validation_summary.to_csv(
    REPORT_TABLES_DIR / "schema_validation_summary.csv",
    index=False,
)
schema_validation_summary

,validation_area,question,status,missing_columns,unusable_columns
0,Core schema,Are the minimum complaint-level columns available?,Passed,None,None
1,Target feasibility inputs,Are the fields needed to evaluate target feasibility available?,Passed,None,None
2,Candidate feature availability,Which potential creation-time features are available?,Passed,None,Not assessed at this stage
3,Supporting metadata,Which descriptive or consistency-check columns are available?,Passed,None,Not assessed at this stage


In [6]:
validation_display_groups = [
    (
        "Core schema validation",
        "Are the minimum complaint-level columns available?",
        "core_required",
    ),
    (
        "Target-input availability",
        "Are the fields needed to evaluate target feasibility available?",
        "target_construction",
    ),
    (
        "Candidate-feature availability",
        "Which potential creation-time features are available?",
        "candidate_feature",
    ),
    (
        "Supporting metadata availability",
        "Which descriptive or consistency-check columns are available?",
        "supporting_metadata",
    ),
]

for heading, question, group_name in validation_display_groups:
    display(Markdown(f"### {heading}\n\n{question}"))
    display(
        column_validation.loc[
            column_validation["column_group"] == group_name
        ].reset_index(drop=True)
    )

### Core schema validation

Are the minimum complaint-level columns available?

,column_name,column_group,is_present,severity_if_missing,impact_if_missing
0,unique_key,core_required,True,critical,The dataset cannot reliably support complaint-level analysis.
1,created_date,core_required,True,critical,The dataset cannot reliably support complaint-level analysis.
2,agency,core_required,True,critical,The dataset cannot reliably support complaint-level analysis.
3,complaint_type,core_required,True,critical,The dataset cannot reliably support complaint-level analysis.


### Target-input availability

Are the fields needed to evaluate target feasibility available?

,column_name,column_group,is_present,severity_if_missing,impact_if_missing
0,closed_date,target_construction,True,critical_for_target,"General analysis may remain possible, but the current missed-resolution-target use case cannot be supported."
1,due_date,target_construction,True,critical_for_target,"General analysis may remain possible, but the current missed-resolution-target use case cannot be supported."
2,status,target_construction,True,critical_for_target,"General analysis may remain possible, but the current missed-resolution-target use case cannot be supported."


### Candidate-feature availability

Which potential creation-time features are available?

,column_name,column_group,is_present,severity_if_missing,impact_if_missing
0,agency,candidate_feature,True,warning,This specific modelling or subgroup-analysis option is reduced.
1,complaint_type,candidate_feature,True,warning,This specific modelling or subgroup-analysis option is reduced.
2,descriptor,candidate_feature,True,warning,This specific modelling or subgroup-analysis option is reduced.
3,borough,candidate_feature,True,warning,This specific modelling or subgroup-analysis option is reduced.
4,open_data_channel_type,candidate_feature,True,warning,This specific modelling or subgroup-analysis option is reduced.


### Supporting metadata availability

Which descriptive or consistency-check columns are available?

,column_name,column_group,is_present,severity_if_missing,impact_if_missing
0,agency_name,supporting_metadata,True,informational,Reporting or consistency-check capability is reduced.


## 6. Issue register

Issues describe the consequence at the correct level. In particular,
unavailable candidate fields limit only specific analyses, while missing
or unusable target timestamps block the current target definition without
making the whole NYC 311 dataset useless.

In [7]:
def build_issue_register() -> pd.DataFrame:
    """Create actionable issues from computed validation results."""
    issues: list[dict[str, str]] = []

    for column in core_missing:
        issues.append(
            {
                "severity": "critical",
                "category": "core_schema",
                "column_name": column,
                "issue": f"Core complaint-level column '{column}' is missing.",
                "impact": (
                    "The dataset cannot reliably support complaint-level analysis."
                ),
            }
        )

    for column in sorted(set(target_missing + target_unusable)):
        if column in {"closed_date", "due_date"}:
            impact = (
                "The dataset may remain usable for general NYC 311 analysis, "
                "but it cannot support the current "
                "missed-resolution-target definition."
            )
        else:
            impact = (
                "Target-feasibility quality checks are incomplete; "
                "general analysis may remain possible."
            )
        state = "missing" if column in target_missing else "unusable"
        issues.append(
            {
                "severity": "critical_for_target",
                "category": "target_feasibility",
                "column_name": column,
                "issue": (
                    f"Target-construction column '{column}' is {state}."
                ),
                "impact": impact,
            }
        )

    for column in candidate_missing:
        if column in CORE_REQUIRED_COLUMNS:
            continue
        issues.append(
            {
                "severity": "warning",
                "category": "candidate_feature",
                "column_name": column,
                "issue": f"Candidate feature '{column}' is unavailable.",
                "impact": (
                    f"Analyses or future models using '{column}' are limited; "
                    "the core dataset is not invalidated."
                ),
            }
        )

    for column in supporting_missing:
        issues.append(
            {
                "severity": "informational",
                "category": "supporting_metadata",
                "column_name": column,
                "issue": f"Supporting field '{column}' is unavailable.",
                "impact": (
                    "Reporting or consistency checks are reduced; "
                    "dataset validity is unchanged."
                ),
            }
        )

    if "due_date" in dataset_columns:
        issues.append(
            {
                "severity": "review_required",
                "category": "prediction_time_availability",
                "column_name": "due_date",
                "issue": (
                    "Confirm that due_date is assigned and available at complaint creation."
                ),
                "impact": (
                    "It must not be used as a prediction-time input until "
                    "source-system timing is verified."
                ),
            }
        )

    if not issues:
        issues.append(
            {
                "severity": "informational",
                "category": "validation",
                "column_name": "",
                "issue": (
                    "No missing-column issues were detected in the classified fields."
                ),
                "impact": (
                    "Further missingness, consistency, and leakage analysis is still required."
                ),
            }
        )

    issue_frame = pd.DataFrame(issues)
    issue_frame.insert(
        0,
        "issue_id",
        [f"DQ-{index:03d}" for index in range(1, len(issue_frame) + 1)],
    )
    return issue_frame


issue_register = build_issue_register()
issue_register.to_csv(
    REPORT_TABLES_DIR / "data_quality_issue_register.csv",
    index=False,
)
issue_register

,issue_id,severity,category,column_name,issue,impact
0,DQ-001,review_required,prediction_time_availability,due_date,Confirm that due_date is assigned and available at complaint creation.,It must not be used as a prediction-time input until source-system timing is verified.


## 7. Generated data-quality report

The report below is assembled from the role definitions and computed
validation results. Dataset-specific counts and statuses are not manually
embedded.

In [8]:
def markdown_table(frame: pd.DataFrame) -> str:
    """Render a small dataframe as a dependency-free Markdown table."""
    display_frame = frame.fillna("").astype(str)
    headers = [str(column) for column in display_frame.columns]

    def clean(value: str) -> str:
        return value.replace("|", "\\|").replace("\n", " ")

    lines = [
        "| " + " | ".join(map(clean, headers)) + " |",
        "| " + " | ".join(["---"] * len(headers)) + " |",
    ]
    lines.extend(
        "| "
        + " | ".join(clean(value) for value in row)
        + " |"
        for row in display_frame.itertuples(index=False, name=None)
    )
    return "\n".join(lines)


report_role_columns = [
    "column_name",
    "column_group",
    "role",
    "required_for_current_stage",
    "available_at_prediction_time",
    "potential_leakage",
]
report_status_columns = [
    "validation_area",
    "status",
    "missing_columns",
    "unusable_columns",
]
report_issue_columns = [
    "issue_id",
    "severity",
    "column_name",
    "issue",
    "impact",
]

quality_report = f"""# NYC 311 Data-Quality Report

Generated by `notebooks/02_schema_and_quality_analysis.ipynb` at
{retrieved_at_utc.isoformat()}.

## Scope

This report assesses schema and data quality for the current Month 1
business question: predicting at complaint creation whether a request
will miss its expected resolution target. It does not build the target,
clean data, impute values, remove columns, select final features, or train
a model.

## Column roles and requirement levels

Structurally mandatory fields identify a complaint, establish its
creation time, and provide its principal agency and complaint category.
Target-dependent fields support the proposed comparison of actual closure
with the expected deadline. Candidate features are possibilities for
later modelling or subgroup analysis, not guaranteed model inputs.
Supporting metadata improves interpretation and consistency checks.

{markdown_table(column_role_summary[report_role_columns])}

A missing `due_date` blocks the current missed-resolution-target
definition because there is no expected deadline to compare with actual
closure. That limitation does not make the complete dataset useless:
complaint-volume, category, geography, channel, and other general NYC 311
analyses may still be possible when their own required fields are valid.

`closed_date`, final `status`, and other post-creation values must not be
used as prediction-time features because they reveal outcome or workflow
information that is unavailable when a new complaint is created.
`due_date` is target-dependent, and its prediction-time availability must
be confirmed from source-system timing before any feature decision.

## Validation summary

{markdown_table(schema_validation_summary[report_status_columns])}

These statuses are intentionally separate. Missing candidate features do
not fail the core schema, and target-input failure does not automatically
invalidate the dataset for unrelated descriptive analysis.

## Important-field completeness

- Source row count at analysis time: {total_row_count:,}
- `closed_date` non-null count: {non_null_counts['closed_date']:,}
- `due_date` non-null count: {non_null_counts['due_date']:,}
- `status` non-null count: {non_null_counts['status']:,}

Timestamp usability is based on the bounded deterministic sample displayed
in the notebook and should be reassessed for the final scoped extract.

## Issue register

{markdown_table(issue_register[report_issue_columns])}

## Boundaries and next checks

Candidate features still require later validation for creation-time
availability, leakage, missingness, stability, and predictive value.
Final target eligibility rules, unresolved or censored requests, scope,
and the final feature list are outside this notebook.
"""
quality_report = "\n".join(
    line[8:] if line.startswith("        ") else line
    for line in quality_report.strip().splitlines()
) + "\n"
QUALITY_REPORT_PATH.write_text(quality_report, encoding="utf-8")

display(Markdown(quality_report))

# NYC 311 Data-Quality Report

Generated by `notebooks/02_schema_and_quality_analysis.ipynb` at
2026-07-24T07:39:39.337367+00:00.

## Scope

This report assesses schema and data quality for the current Month 1
business question: predicting at complaint creation whether a request
will miss its expected resolution target. It does not build the target,
clean data, impute values, remove columns, select final features, or train
a model.

## Column roles and requirement levels

Structurally mandatory fields identify a complaint, establish its
creation time, and provide its principal agency and complaint category.
Target-dependent fields support the proposed comparison of actual closure
with the expected deadline. Candidate features are possibilities for
later modelling or subgroup analysis, not guaranteed model inputs.
Supporting metadata improves interpretation and consistency checks.

| column_name | column_group | role | required_for_current_stage | available_at_prediction_time | potential_leakage |
| --- | --- | --- | --- | --- | --- |
| unique_key | core dataset requirement | identifier | yes | yes | no |
| created_date | core dataset requirement | event timestamp | yes | yes | no |
| closed_date | target construction | outcome timestamp | needed for target feasibility analysis | no | yes, if used as a model feature |
| due_date | target construction | expected resolution deadline | needed for target feasibility analysis | must be confirmed | depends on whether it exists at complaint creation |
| status | target construction | target-supporting workflow state | needed for target feasibility quality checks | initial status may be available; final status is not | yes, when final status is used |
| agency | core requirement and candidate feature | responsible organization | yes | yes | no |
| agency_name | supporting metadata | human-readable agency label | no | usually yes | no |
| complaint_type | core requirement and candidate feature | main request category | yes | yes | no |
| descriptor | candidate feature | detailed complaint category | no | generally yes | no, subject to source validation |
| borough | candidate feature | geographic grouping | no | generally yes | no |
| open_data_channel_type | candidate feature | complaint intake channel | no | yes | no |

A missing `due_date` blocks the current missed-resolution-target
definition because there is no expected deadline to compare with actual
closure. That limitation does not make the complete dataset useless:
complaint-volume, category, geography, channel, and other general NYC 311
analyses may still be possible when their own required fields are valid.

`closed_date`, final `status`, and other post-creation values must not be
used as prediction-time features because they reveal outcome or workflow
information that is unavailable when a new complaint is created.
`due_date` is target-dependent, and its prediction-time availability must
be confirmed from source-system timing before any feature decision.

## Validation summary

| validation_area | status | missing_columns | unusable_columns |
| --- | --- | --- | --- |
| Core schema | Passed | None | None |
| Target feasibility inputs | Passed | None | None |
| Candidate feature availability | Passed | None | Not assessed at this stage |
| Supporting metadata | Passed | None | Not assessed at this stage |

These statuses are intentionally separate. Missing candidate features do
not fail the core schema, and target-input failure does not automatically
invalidate the dataset for unrelated descriptive analysis.

## Important-field completeness

- Source row count at analysis time: 21,911,249
- `closed_date` non-null count: 21,496,251
- `due_date` non-null count: 76,953
- `status` non-null count: 21,911,249

Timestamp usability is based on the bounded deterministic sample displayed
in the notebook and should be reassessed for the final scoped extract.

## Issue register

| issue_id | severity | column_name | issue | impact |
| --- | --- | --- | --- | --- |
| DQ-001 | review_required | due_date | Confirm that due_date is assigned and available at complaint creation. | It must not be used as a prediction-time input until source-system timing is verified. |

## Boundaries and next checks

Candidate features still require later validation for creation-time
availability, leakage, missingness, stability, and predictive value.
Final target eligibility rules, unresolved or censored requests, scope,
and the final feature list are outside this notebook.


## 8. Scope confirmation

This notebook validates, classifies, analyses, and reports only. It does
not create `missed_resolution_target`, train a model, clean or remove
records, impute missing values, or decide the final feature set.